# 11. Customer Support Handoffs — Multi-agent state transitions

Customer support agents often need to move a case between general support, billing, technical support, and human approval. This example models handoffs as explicit state transitions.

**Learning goals**
- Define handoff criteria for support workflows.
- Represent ownership changes as graph transitions.
- Add approval gates for sensitive actions.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class SupportState(TypedDict):
    message: str
    owner: str
    resolution: str

## 11.1 Handoff criteria

Handoffs should be based on clear signals such as refund requests, technical errors, or escalation language. Ambiguous ownership creates slow and inconsistent support.


In [ ]:
def triage(state: SupportState) -> dict:
    msg = state["message"].lower()
    if "refund" in msg or "refund" in msg:
        return {"owner": "billing"}
    if "error" in msg or "error" in msg:
        return {"owner": "technical"}
    return {"owner": "general"}

In [ ]:
def resolve(state: SupportState) -> dict:
    templates = {
        "billing": "Check the refund policy and prepare an approval request.",
        "technical": "Ask for reproduction steps and logs.",
        "general": "Provide the standard guidance.",
    }
    return {"resolution": templates[state["owner"]]}

## 11.2 Represent the flow as a graph

A graph makes ownership and transition rules visible. Each route can be tested without relying on a full support transcript.


In [ ]:
builder = StateGraph(SupportState)
builder.add_node("triage", triage)
builder.add_node("resolve", resolve)
builder.add_edge(START, "triage")
builder.add_edge("triage", "resolve")
builder.add_edge("resolve", END)
graph = builder.compile()

In [ ]:
result = graph.invoke({
    "message": "I would like a subscription refund.",
    "owner": "", "resolution": "",
})
result

## 11.3 Approval gate

Some support actions require human confirmation. An approval gate records why the case is paused and what decision is needed next.


In [ ]:
def needs_approval(state: SupportState) -> bool:
    return state["owner"] == "billing"

print("approval required:", needs_approval(result))

---

## Summary

| Item | Content |
|---|---|
| **Covered** | handoff state machines, escalation, owner transitions, and approval gates |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`handoffs.md`](../../docs/langchain/multi-agent/handoffs.md)
- [`interrupts.md`](../../docs/langgraph/interrupts.md)
- [`human-in-the-loop.md`](../../docs/langchain/human-in-the-loop.md)
